# 第 1 周末练习 —— 技术问答解释器（OpenAI + Ollama）

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（例如「初级工程师要成为合格 AI Engineer，该练哪些能力？」）
- **输出**：清晰、可执行的学习指导
- **额外要求**：用**流式（streaming）**一边生成一边更新显示

这是你在课程期间自己也能天天用的工具。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 云端模型 | `gpt-4o-mini`（`MODEL_GPT`），经 OpenRouter 的 `base_url` |
| 本地 Ollama | `llama3.2`（`MODEL_LLAMA`），OpenAI 兼容 `/v1` |
| 流式输出 | `client.responses.stream(...)` + `update_display` |
| 环境变量 | `OPENAI_API_KEY`、`OPENROUTER_BASE_URL` |

## 怎么跑

1. 准备 `.env`：`OPENAI_API_KEY`、`OPENROUTER_BASE_URL`；本地需 Ollama 已拉取 `llama3.2`
2. 从上到下运行；在「提问」格改 `input`，再分别跑 GPT 与 Llama 两格对比


In [ ]:
# ========== 导入：后面要用的工具箱 ==========

# os：读环境变量（API Key、Base URL）
import os
# OpenAI 客户端：本练习既用来打 OpenRouter，也用来打 Ollama 的 OpenAI 兼容接口
from openai import OpenAI
# load_dotenv / find_dotenv：定位并加载 .env，避免密钥写进代码
from dotenv import load_dotenv, find_dotenv
# Markdown 展示 + display_display：流式时不断刷新同一块输出
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 常量 + 从环境变量读密钥与网关地址 ==========

# OpenAI 兼容云端小模型名（经 OpenRouter 等网关时，名字需网关认识）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2
MODEL_LLAMA = 'llama3.2'

# 加载 .env：find_dotenv() 向上找文件；override=True 覆盖已有环境变量
load_dotenv(find_dotenv(), override=True)
# 读取 API Key（Environment Variable 名保持 OPENAI_API_KEY）
api_key = os.getenv("OPENAI_API_KEY")
# 读取 OpenRouter（或兼容网关）的 Base URL
base_url = os.getenv("OPENROUTER_BASE_URL")


In [ ]:
# ========== 启动前检查：缺配置就打印提示（文案原样保留） ==========

# 若 api_key 为空，提示用户去环境变量里设置（print 字符串勿改译，可能被脚本依赖）
if not api_key:
    print("OPENAI_API_KEY is not set in the environment variables")

# 若 base_url 为空，同样提示（OpenRouter 网关地址缺失）
if not base_url:
    print("OPENROUTER_BASE_URL is not set in the environment variables")


In [ ]:
# ========== 初始化两个 OpenAI 兼容客户端 ==========

# 云端/网关客户端：base_url 指向 OpenRouter，密钥用上面读到的 api_key
openai = OpenAI(
    base_url=base_url,
    api_key=api_key
)

# 本地 Ollama：走 OpenAI 兼容的 /v1；api_key 占位字符串 "ollama" 即可（本地常不校验）
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)



In [ ]:
# ========== 提问内容：system 级 instruction + user 问题 ==========

# 在此改写 input 即可换问题；instruction 是发给模型的角色设定（英文保留）

# instructions：Responses API 里的「系统级」指导（相当于旧版 system message）
instruction = """
You are a professional software engineer with expertise in Python programming, Data Science, and AI Engineering.
You are tasked with guiding junior engineers in their learning journey to become proficient AI Engineers.
"""

# input：用户真正要问的问题（变量名就叫 input，保持原样，勿改名）
input = "What are the key skills and knowledge areas that a junior engineer should focus on to become a proficient AI Engineer?"


In [ ]:
# ========== 流式工具函数：Responses API + 笔记本增量刷新 ==========

# 使用较新的 Responses API（client.responses.stream），而不是旧的 chat.completions
# 它是官方更推荐的流式接口形态之一；事件里用 delta 拼全文

def get_response(model):
    """按 model 名选客户端，流式生成回答，并用 update_display 边收边刷新 Markdown。"""
    # 若是本地 Llama 就用 ollama 客户端，否则用云端 openai 客户端
    client = ollama if model == MODEL_LLAMA else openai
    # with：确保流结束后正确关闭连接；stream 事件迭代
    with client.responses.stream(
        model=model,
        instructions=instruction,
        input=input
    ) as stream:
        # 累加已收到的全文，供每次刷新显示
        response = ""
        # 先放一块空 Markdown，拿到 display_id，后续原地更新
        display_handle = display(Markdown(""), display_id=True)
        # 遍历流式事件
        for event in stream:
            # 只处理「输出文本增量」类事件
            if event.type == "response.output_text.delta":
                # 把本片 delta 拼到全文
                response += event.delta
                # 用同一 display_id 刷新，实现打字机效果
                update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 调用：用云端 gpt-4o-mini 流式回答 ==========

# 传入 MODEL_GPT，走 OpenRouter/云端客户端
get_response(MODEL_GPT)


In [ ]:
# ========== 调用：用本地 Llama 3.2 流式回答 ==========

# 传入 MODEL_LLAMA，走本机 Ollama（需服务已启动且已 pull 模型）
get_response(MODEL_LLAMA)
